# Day 27: CNN vs ViT — Head-to-Head Comparison

> **Goal:** Train a CNN and a ViT with similar parameter counts on CIFAR-10. Compare accuracy, training speed, data efficiency, and interpretability.

---

## The Big Question

CNNs have strong **inductive biases** (locality, translation equivariance). ViTs have **no** built-in biases — they must learn everything from data.

| Property | CNN | ViT |
|---|---|---|
| **Inductive bias** | Strong (local filters) | Weak (global attention) |
| **Data efficiency** | Better with less data | Needs more data |
| **Scalability** | Saturates at scale | Improves with more data |
| **Computation** | O(k² · H · W) | O(N² · d) |
| **Feature type** | Hierarchical local → global | Global from layer 1 |

Today we'll test these claims empirically!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import math
import time

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Define Both Architectures

In [ ]:
# ============================================================
# CNN (ResNet-style)
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        self.layer1 = nn.Sequential(ConvBlock(64, 64), ConvBlock(64, 64))
        self.layer2 = nn.Sequential(ConvBlock(64, 128, stride=2), ConvBlock(128, 128))
        self.layer3 = nn.Sequential(ConvBlock(128, 256, stride=2), ConvBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

### CNN Architecture: ResNet-Style

The CNN below uses **residual blocks** (skip connections), which allow gradients to flow directly through the network. This design is the backbone of most modern CNNs.

Key properties of CNNs:
- **Local receptive fields:** Each conv filter only looks at a small patch (3×3)
- **Parameter sharing:** The same filter slides across the entire image
- **Translation equivariance:** If the object moves, the feature map shifts accordingly
- **Hierarchical features:** Early layers detect edges → middle layers detect textures → deep layers detect objects

In [ ]:
# ============================================================
# ViT
# ============================================================
class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, num_classes=10,
                 d_model=128, num_heads=4, num_layers=6, dropout=0.1):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2
        self.patch_proj = nn.Conv2d(in_channels, d_model, patch_size, patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model) * 0.02)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=d_model*4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, num_classes))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_proj(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.dropout(x + self.pos_embed)
        x = self.norm(self.encoder(x))
        return self.head(x[:, 0])

### ViT Architecture

In contrast, the ViT has **no built-in spatial biases**. It treats the image as a sequence of patches and uses self-attention to learn relationships between them. This means:
- Every patch can attend to every other patch from layer 1 (global context)
- Spatial relationships must be **learned** from data via positional embeddings
- It needs **more data** and **stronger regularization** to compensate for the lack of inductive biases

### Parameter Count Comparison

For a **fair comparison**, both models should have roughly similar parameter counts. If one model has 10× more parameters, any performance difference could simply be due to model capacity rather than architectural advantages. Let's check:

In [ ]:
# Compare parameter counts
cnn = CNN()
vit = ViT()
cnn_params = sum(p.numel() for p in cnn.parameters())
vit_params = sum(p.numel() for p in vit.parameters())

print(f"CNN parameters: {cnn_params:>10,}")
print(f"ViT parameters: {vit_params:>10,}")
print(f"Ratio (ViT/CNN): {vit_params/cnn_params:.2f}×")

---
## 2. Data Setup

In [ ]:
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.2)])

test_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
train_data, val_data = random_split(full_train, [45000, 5000], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=128, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=128, shuffle=False)
classes = full_train.classes

---
## 3. Training Framework

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def train_model(model, name, epochs=40, lr=3e-4, warmup=5, weight_decay=0.05,
                label_smoothing=0.1, train_loader=train_loader):
    """Train a model and return history."""
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Warmup + cosine decay
    def lr_lambda(epoch):
        if epoch < warmup:
            return epoch / warmup
        return 0.5 * (1 + math.cos(math.pi * (epoch - warmup) / (epochs - warmup)))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": [], "time": []}
    best_val_acc = 0

    for epoch in range(epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        vl, va = evaluate(model, val_loader, loss_fn, device)
        scheduler.step()
        elapsed = time.time() - t0

        history["train_acc"].append(ta); history["val_acc"].append(va)
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["time"].append(elapsed)

        if va > best_val_acc:
            best_val_acc = va

        if (epoch+1) % 10 == 0:
            print(f"  {name} Epoch {epoch+1}: Train {ta:.2%}, Val {va:.2%} ({elapsed:.1f}s)")

    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
    print(f"  {name} ✅ Test: {test_acc:.2%} | Best Val: {best_val_acc:.2%}")
    history["test_acc"] = test_acc
    history["best_val"] = best_val_acc
    return history

---
## 4. Head-to-Head: Full Dataset

In [ ]:
EPOCHS = 40

print("=" * 50)
print("CNN Training")
print("=" * 50)
torch.manual_seed(42)
cnn_model = CNN().to(device)
h_cnn = train_model(cnn_model, "CNN", epochs=EPOCHS, lr=1e-3, warmup=0,
                     weight_decay=1e-4, label_smoothing=0.0)

print("\n" + "=" * 50)
print("ViT Training")
print("=" * 50)
torch.manual_seed(42)
vit_model = ViT().to(device)
h_vit = train_model(vit_model, "ViT", epochs=EPOCHS, lr=3e-4, warmup=5,
                     weight_decay=0.05, label_smoothing=0.1)

### Head-to-Head Training

Notice the **different training recipes** for each model:

| Setting | CNN | ViT |
|---|---|---|
| Learning rate | 1e-3 (higher) | 3e-4 (lower) |
| Warmup epochs | 0 (none needed) | 5 (essential for stability) |
| Weight decay | 1e-4 (light) | 0.05 (heavy — ViTs need stronger regularization) |
| Label smoothing | 0.0 (not needed) | 0.1 (helps prevent overconfidence) |

ViTs are more sensitive to hyperparameters than CNNs — they require warmup, stronger regularization, and often heavier data augmentation to train well.

In [ ]:
# Compare training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(h_cnn["train_loss"], label="CNN Train", color="C0")
axes[0].plot(h_cnn["val_loss"], "--", label="CNN Val", color="C0")
axes[0].plot(h_vit["train_loss"], label="ViT Train", color="C1")
axes[0].plot(h_vit["val_loss"], "--", label="ViT Val", color="C1")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(h_cnn["train_acc"], label="CNN Train", color="C0")
axes[1].plot(h_cnn["val_acc"], "--", label="CNN Val", color="C0")
axes[1].plot(h_vit["train_acc"], label="ViT Train", color="C1")
axes[1].plot(h_vit["val_acc"], "--", label="ViT Val", color="C1")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Time per epoch
axes[2].bar(["CNN", "ViT"],
            [np.mean(h_cnn["time"]), np.mean(h_vit["time"])],
            color=["C0", "C1"])
axes[2].set_title("Avg Time per Epoch (s)")
axes[2].grid(True, alpha=0.3, axis="y")

for ax in axes[:2]: ax.set_xlabel("Epoch")
plt.suptitle("CNN vs ViT on CIFAR-10 (Full Dataset)", fontsize=14)
plt.tight_layout(); plt.show()

# Summary
print(f"\n{'Metric':<20} {'CNN':>10} {'ViT':>10}")
print("-" * 42)
print(f"{'Test Accuracy':<20} {h_cnn['test_acc']:>9.2%} {h_vit['test_acc']:>9.2%}")
print(f"{'Best Val Accuracy':<20} {h_cnn['best_val']:>9.2%} {h_vit['best_val']:>9.2%}")
print(f"{'Avg Epoch Time':<20} {np.mean(h_cnn['time']):>8.1f}s {np.mean(h_vit['time']):>8.1f}s")
print(f"{'Parameters':<20} {sum(p.numel() for p in CNN().parameters()):>10,} {sum(p.numel() for p in ViT().parameters()):>10,}")

### Reading the Comparison Plots

In the plots below, look for:
- **Loss curves:** Does one model converge faster? Is there a gap between train and val loss (indicating overfitting)?
- **Accuracy curves:** Does the CNN plateau earlier while the ViT keeps improving? Or vice versa?
- **Training speed:** The bar chart shows wall-clock time per epoch — ViTs are often slower due to self-attention's O(N²) complexity.

---
## 5. Data Efficiency Experiment

The key claim: CNNs are more data-efficient due to their inductive biases.

In [ ]:
# Train both on subsets of different sizes
subset_sizes = [500, 2000, 10000, 45000]
results = {"CNN": [], "ViT": []}

for size in subset_sizes:
    print(f"\n{'='*40}")
    print(f"Training with {size} samples")
    print(f"{'='*40}")

    # Create subset
    indices = torch.randperm(len(train_data), generator=torch.Generator().manual_seed(42))[:size]
    subset = Subset(train_data, indices.tolist())
    sub_loader = DataLoader(subset, batch_size=min(128, size), shuffle=True)

    # CNN
    torch.manual_seed(42)
    cnn_sub = CNN().to(device)
    h = train_model(cnn_sub, f"CNN-{size}", epochs=30, lr=1e-3, warmup=0,
                    weight_decay=1e-4, label_smoothing=0.0, train_loader=sub_loader)
    results["CNN"].append(h["test_acc"])

    # ViT
    torch.manual_seed(42)
    vit_sub = ViT().to(device)
    h = train_model(vit_sub, f"ViT-{size}", epochs=30, lr=3e-4, warmup=3,
                    weight_decay=0.05, label_smoothing=0.1, train_loader=sub_loader)
    results["ViT"].append(h["test_acc"])

### How This Experiment Works

We train both models on **progressively larger subsets** (500, 2000, 10000, 45000 samples) and compare test accuracy at each size. This directly tests the hypothesis that CNNs are more data-efficient:

- With **500 samples**, the CNN's inductive biases (locality, translation equivariance) should give it a big advantage — it "knows" that nearby pixels are related even without seeing many examples.
- With **45,000 samples** (full CIFAR-10), the ViT has enough data to learn these spatial relationships on its own, so the gap should narrow.

⚠️ **This cell takes a while to run** — it trains 8 models (2 architectures × 4 dataset sizes).

In [ ]:
# Plot data efficiency
plt.figure(figsize=(8, 5))
plt.plot(subset_sizes, results["CNN"], "o-", label="CNN", markersize=8, linewidth=2)
plt.plot(subset_sizes, results["ViT"], "s-", label="ViT", markersize=8, linewidth=2)
plt.xscale("log")
plt.xlabel("Training Set Size")
plt.ylabel("Test Accuracy")
plt.title("Data Efficiency: CNN vs ViT")
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(subset_sizes, [str(s) for s in subset_sizes])

# Add gap annotations
for i, size in enumerate(subset_sizes):
    gap = results["CNN"][i] - results["ViT"][i]
    mid = (results["CNN"][i] + results["ViT"][i]) / 2
    plt.annotate(f"Δ={gap:+.1%}", xy=(size, mid), fontsize=8, ha="left")

plt.tight_layout()
plt.show()

### Interpreting the Data Efficiency Curve

The plot below uses a **log scale** on the x-axis (training set size). Look for:
- The **Δ annotations** showing the accuracy gap at each dataset size
- Whether the gap **shrinks** as you move right (more data) — this is the expected pattern
- At what dataset size (if any) the ViT catches up with or surpasses the CNN

---
## 6. Analysis & Discussion

In [ ]:
# Summary table
print(f"\n{'Dataset Size':>12} | {'CNN Acc':>8} | {'ViT Acc':>8} | {'Winner':>8} | {'Gap':>8}")
print("-" * 55)
for i, size in enumerate(subset_sizes):
    cnn_acc = results["CNN"][i]
    vit_acc = results["ViT"][i]
    winner = "CNN" if cnn_acc > vit_acc else "ViT"
    gap = abs(cnn_acc - vit_acc)
    print(f"{size:>12,} | {cnn_acc:>7.2%} | {vit_acc:>7.2%} | {winner:>8} | {gap:>7.2%}")

print("\n📊 Key observations:")
print("• CNNs typically win with small datasets (strong inductive biases help)")
print("• ViTs catch up or surpass CNNs with more data")
print("• On ImageNet-scale (1.2M images), ViTs dominate")
print("• On CIFAR-10 (50K images), the gap depends on model size and training tricks")

---
## 🧪 Exercises

### Exercise 1: Hybrid model
Build a model that uses CNN layers for initial feature extraction, then applies Transformer layers. Does it combine the best of both?

### Exercise 2: Same parameters
Adjust the ViT (fewer layers/smaller d_model) to have exactly the same parameter count as the CNN. Re-run the comparison.

### Exercise 3: Augmentation ablation
Train both models without any augmentation. Which suffers more?

### Exercise 4: Per-class comparison
Compute per-class accuracy for both models. Are there classes where one model is clearly better?

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Finding | Detail |
|---|---|
| **CNN advantage** | Better with small data, faster per epoch, strong locality bias |
| **ViT advantage** | More flexible, scales better with data, global attention |
| **Data efficiency** | CNN >> ViT on small datasets; gap narrows with more data |
| **Training** | ViTs need warmup, label smoothing, stronger augmentation |
| **Inductive bias** | Helpful when data is scarce, can limit ceiling with big data |
| **In practice** | Many SOTA models are hybrids (ConvNeXt, Swin Transformer) |

**Tomorrow:** Advanced Architectures — exploring hybrids, ConvNeXt, Swin Transformers, and modern design choices!